**Table of contents**<a id='toc0_'></a>    
- [Imports](#toc1_)    
- [Constants](#toc2_)    
- [Paths](#toc3_)    
- [Helper functions](#toc4_)    
- [Model definations](#toc5_)    
- [Traditional + Augumented](#toc6_)    
    - [Update path](#toc6_1_1_)    
    - [Load data](#toc6_1_2_)    
    - [Train models](#toc6_1_3_)    
      - [Unet + Resnet50](#toc6_1_3_1_)    
      - [Deepab + Resnet50](#toc6_1_3_2_)    
      - [Unet + Resnet101](#toc6_1_3_3_)    
      - [Deeplab + Resnet101](#toc6_1_3_4_)    
- [Traditional](#toc7_)    
    - [Update path](#toc7_1_1_)    
    - [Load data](#toc7_1_2_)    
    - [Train models](#toc7_1_3_)    
      - [Unet + Resnet50](#toc7_1_3_1_)    
      - [Deepab + Resnet50](#toc7_1_3_2_)    
      - [Unet + Resnet101](#toc7_1_3_3_)    
      - [Deeplab + Resnet101](#toc7_1_3_4_)    
- [Traditional + Sythetic](#toc8_)    
    - [Update path](#toc8_1_1_)    
    - [Load data](#toc8_1_2_)    
    - [Train models](#toc8_1_3_)    
      - [Unet + Resnet50](#toc8_1_3_1_)    
      - [Deepab + Resnet50](#toc8_1_3_2_)    
      - [Unet + Resnet101](#toc8_1_3_3_)    
      - [Deeplab + Resnet101](#toc8_1_3_4_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc1_'></a>[Imports](#toc0_)

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import segmentation_models_pytorch as smp

import numpy as np

from sklearn.model_selection import train_test_split 
import os


# <a id='toc2_'></a>[Constants](#toc0_)

In [ ]:
DEVICE  = torch.device(f"cuda:0" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 10
EPOCHS = 25
print(DEVICE)


cpu


# <a id='toc3_'></a>[Paths](#toc0_)

In [71]:
base_path = "../Dataset"

dataset = base_path + "/s2-utm-33N-18E-242N-2018"
train_geojson_path = base_path + "/br-18E-242N-crop-labels-train-2018.geojson"

folder = "/DS3"
subfolder1 = "/Temporal Data"

data_path = dataset+folder
temporal_data_path = dataset+folder+subfolder1

classification_model_path = "../Classification models/Spatial"
temporal_classification_model_path = "../Classification models/Temporal"


# <a id='toc4_'></a>[Helper functions](#toc0_)

In [72]:
class ImageLabelDataset(Dataset):
    def __init__(self, image_array, label_array):
        self.images = torch.tensor(image_array, dtype=torch.float32)  # (N, 4, 64, 64)
        self.labels = torch.tensor(label_array, dtype=torch.float32)  # (N, 1, 64, 64)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]


In [73]:
def save_models(
    model,
    optimizer,
    scheduler,
    scaler,
    epoch,
    segmentation_model_name,
    encoder_name,
    path
):
    filename = f'/Classification_Model_Pipeline-{segmentation_model_name}_Encoder-{encoder_name}'+r'-Epoch '+str(epoch)+r'.pth'
    full_path = path + filename

    payload = {
        "epochs": int(epoch),
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state": scaler.state_dict(),
    }
    torch.save(payload, full_path)

    print(f"Models saved to {full_path}")

def load_models(
    path,
    model,
    optimizer,
    scheduler,
    scaler,
    device="cpu",
):
    """
    Load a training checkpoint. Returns the next epoch index to continue from.
    Loads only what you pass in (model, optimizer, scheduler, scaler).
    """
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    scheduler.load_state_dict(checkpoint["scheduler_state"])
    scaler.load_state_dict(checkpoint["scaler_state"])

    start_epoch = int(checkpoint.get("epochs"))
    return start_epoch


# <a id='toc5_'></a>[Model definations](#toc0_)

In [74]:

# U-Net with ResNet-150 encoder
unet_r50 = smp.Unet(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=4,
    classes=NUM_CLASSES
)

# DeepLabV3+ with ResNet-50
deeplab_r50 = smp.DeepLabV3Plus(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=4,
    classes=NUM_CLASSES
)

# U-Net with ResNet-101 encoder
unet_r101 = smp.Unet(
    encoder_name="resnet101",
    encoder_weights=None,
    in_channels=4,
    classes=NUM_CLASSES
)

# DeepLabV3+ with ResNet-101
deeplab_r101 = smp.DeepLabV3Plus(
    encoder_name="resnet101",
    encoder_weights=None,
    in_channels=4,
    classes=NUM_CLASSES
)

model_list = [unet_r50, deeplab_r50, unet_r101, deeplab_r101]
model_name_list = ['unet_r50', 'deeplab_r50', 'unet_r101', 'deeplab_r101']
encoder_name_list = ['resnet50', 'resnet101']


# <a id='toc6_'></a>[Traditional + Augumented](#toc0_)

### <a id='toc6_1_1_'></a>[Update path](#toc0_)

In [ ]:
classification_model_path = "../Classification models/Spatial/Traditional and augmented"


### <a id='toc6_1_2_'></a>[Load data](#toc0_)

In [ ]:
data = np.load('../Dataset/s2-utm-33N-18E-242N-2018/DS3/11_final_gan_data_without_mixed_patches_64X64_majority50.npy')
labels = np.load('../Dataset/s2-utm-33N-18E-242N-2018/DS3/12_final_gan_labels_without_mixed_patches_64X64_majority50.npy')

labels_upadated = np.where(labels == -1, 0, labels)
train_data, validation_data, train_labels, validation_labels = train_test_split(data, labels_upadated, random_state= 2) 

image_data = np.transpose(train_data, (0, 3, 1, 2))  # (N, 4, 64, 64)

dataset = ImageLabelDataset(image_data, train_labels)
loader = DataLoader(dataset, batch_size=32, shuffle=True)


### <a id='toc6_1_3_'></a>[Train models](#toc0_)

#### <a id='toc6_1_3_1_'></a>[Unet + Resnet50](#toc0_)

In [ ]:
selected_model_index = 0
model = model_list[selected_model_index]
segmentation_model_name = model_name_list[selected_model_index]
encoder_name = 'resnet50'if selected_model_index<2 else 'resnet101'
print(segmentation_model_name, encoder_name)

# Loss: CE + Dice (multiclass)
dice = smp.losses.DiceLoss(mode="multiclass")
ce   = nn.CrossEntropyLoss()   # add class weights if needed
def loss_fn(logits, y): return 0.5*ce(logits, y) + 0.5*dice(logits, y)

# Optim & sched
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)

# Mixed precision training loop (sketch)
scaler = torch.amp.GradScaler(DEVICE.type)

for epoch in range(EPOCHS):
    print(f'Epoch: {epoch}')
    epoch_loss = []
    model.train()

    for x, y in loader:       # x:(B,4,64,64) in [-1,1], y:(B,64,64) long
        x, y = x.clone().detach().to(DEVICE), y.clone().detach().long().to(DEVICE)

        with torch.cuda.amp.autocast(DEVICE):
            logits = model(x)
            loss = loss_fn(logits, y)
            epoch_loss.append(loss.item())
        
        
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad()

    print(f'Loss: {loss.item():.4f} | Epoch avg loss: {sum(epoch_loss)/len(epoch_loss)}')
    sched.step()

save_models(model,
            opt, 
            sched, 
            scaler, 
            (epoch+1), 
            segmentation_model_name, 
            encoder_name, 
            classification_model_path)


unet_r50 resnet50
Epoch: 0


C:\Users\2405647\AppData\Local\Temp\ipykernel_25828\2639090813.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(DEVICE):


Loss: 0.9803 | Epoch avg loss: 1.1854789921870599
Epoch: 1
Loss: 1.2236 | Epoch avg loss: 0.9002331667221509
Epoch: 2
Loss: 1.0739 | Epoch avg loss: 0.8007692276285245
Epoch: 3
Loss: 1.3816 | Epoch avg loss: 0.7877344242655314
Epoch: 4
Loss: 0.6755 | Epoch avg loss: 0.7127070426940918
Epoch: 5
Loss: 0.6298 | Epoch avg loss: 0.680980981542514
Epoch: 6
Loss: 0.8152 | Epoch avg loss: 0.6523317410968817
Epoch: 7
Loss: 0.6521 | Epoch avg loss: 0.6469226221625621
Epoch: 8
Loss: 0.7618 | Epoch avg loss: 0.6008656884615238
Epoch: 9
Loss: 0.8576 | Epoch avg loss: 0.5923597683700231
Epoch: 10
Loss: 0.6492 | Epoch avg loss: 0.5524574615634404
Epoch: 11
Loss: 1.0135 | Epoch avg loss: 0.5364400561039264
Epoch: 12
Loss: 1.0370 | Epoch avg loss: 0.5227630333258555
Epoch: 13
Loss: 0.5548 | Epoch avg loss: 0.5137210654524657
Epoch: 14
Loss: 0.4091 | Epoch avg loss: 0.46999066036481124
Epoch: 15
Loss: 0.9220 | Epoch avg loss: 0.43949602630275947
Epoch: 16
Loss: 0.6073 | Epoch avg loss: 0.425819242516389

#### <a id='toc6_1_3_2_'></a>[Deepab + Resnet50](#toc0_)

In [ ]:
selected_model_index = 1
model = model_list[selected_model_index]
segmentation_model_name = model_name_list[selected_model_index]
encoder_name = 'resnet50'if selected_model_index<2 else 'resnet101'
print(segmentation_model_name, encoder_name)

# Loss: CE + Dice (multiclass)
dice = smp.losses.DiceLoss(mode="multiclass")
ce   = nn.CrossEntropyLoss()   # add class weights if needed
def loss_fn(logits, y): return 0.5*ce(logits, y) + 0.5*dice(logits, y)

# Optim & sched
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)

# Mixed precision training loop (sketch)
scaler = torch.amp.GradScaler(DEVICE.type)

for epoch in range(EPOCHS):
    print(f'Epoch: {epoch}')
    epoch_loss = []
    model.train()

    for x, y in loader:       # x:(B,4,64,64) in [-1,1], y:(B,64,64) long
        x, y = x.clone().detach().to(DEVICE), y.clone().detach().long().to(DEVICE)

        with torch.cuda.amp.autocast(DEVICE):
            logits = model(x)
            loss = loss_fn(logits, y)
            epoch_loss.append(loss.item())
        
        
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad()

    print(f'Loss: {loss.item():.4f} | Epoch avg loss: {sum(epoch_loss)/len(epoch_loss)}')
    sched.step()

save_models(model,
            opt, 
            sched, 
            scaler, 
            (epoch+1), 
            segmentation_model_name, 
            encoder_name, 
            classification_model_path)


Epoch: 0


C:\Users\2405647\AppData\Local\Temp\ipykernel_25828\1914483387.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(DEVICE):
c:\Users\2405647\.conda\envs\myenv\lib\site-packages\torch\amp\autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


Loss: 1.0820 | Epoch avg loss: 1.2058267484490688
Epoch: 1
Loss: 0.7749 | Epoch avg loss: 0.9154312066160716
Epoch: 2
Loss: 1.2721 | Epoch avg loss: 0.8265384189211405
Epoch: 3
Loss: 1.1851 | Epoch avg loss: 0.7798352975111741
Epoch: 4
Loss: 0.8794 | Epoch avg loss: 0.7169282447833282
Epoch: 5
Loss: 1.1168 | Epoch avg loss: 0.7024523673149256
Epoch: 6
Loss: 1.2464 | Epoch avg loss: 0.6715920527394001
Epoch: 7
Loss: 1.4771 | Epoch avg loss: 0.6333245273966056
Epoch: 8
Loss: 1.1352 | Epoch avg loss: 0.6571530261291907
Epoch: 9
Loss: 0.9161 | Epoch avg loss: 0.5820433497428894
Epoch: 10
Loss: 0.5373 | Epoch avg loss: 0.5952036893711641
Epoch: 11
Loss: 0.5776 | Epoch avg loss: 0.5382423432400594
Epoch: 12
Loss: 1.0736 | Epoch avg loss: 0.5294359621520226
Epoch: 13
Loss: 0.7589 | Epoch avg loss: 0.5023206902238039
Epoch: 14
Loss: 1.0502 | Epoch avg loss: 0.5136622482767472
Epoch: 15
Loss: 0.8057 | Epoch avg loss: 0.4578948989510536
Epoch: 16
Loss: 0.5208 | Epoch avg loss: 0.4443558007478714

#### <a id='toc6_1_3_3_'></a>[Unet + Resnet101](#toc0_)

In [ ]:
selected_model_index = 2
model = model_list[selected_model_index]
segmentation_model_name = model_name_list[selected_model_index]
encoder_name = 'resnet50'if selected_model_index<2 else 'resnet101'
print(segmentation_model_name, encoder_name)

# Loss: CE + Dice (multiclass)
dice = smp.losses.DiceLoss(mode="multiclass")
ce   = nn.CrossEntropyLoss()   # add class weights if needed
def loss_fn(logits, y): return 0.5*ce(logits, y) + 0.5*dice(logits, y)

# Optim & sched
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)

# Mixed precision training loop (sketch)
scaler = torch.amp.GradScaler(DEVICE.type)

for epoch in range(EPOCHS):
    print(f'Epoch: {epoch}')
    epoch_loss = []
    model.train()

    for x, y in loader:       # x:(B,4,64,64) in [-1,1], y:(B,64,64) long
        x, y = x.clone().detach().to(DEVICE), y.clone().detach().long().to(DEVICE)

        with torch.cuda.amp.autocast(DEVICE):
            logits = model(x)
            loss = loss_fn(logits, y)
            epoch_loss.append(loss.item())
        
        
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad()

    print(f'Loss: {loss.item():.4f} | Epoch avg loss: {sum(epoch_loss)/len(epoch_loss)}')
    sched.step()

save_models(model,
            opt, 
            sched, 
            scaler, 
            (epoch+1), 
            segmentation_model_name, 
            encoder_name, 
            classification_model_path)


Epoch: 0


C:\Users\2405647\AppData\Local\Temp\ipykernel_25828\3051063026.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(DEVICE):


Loss: 0.9072 | Epoch avg loss: 0.6868304154620721
Epoch: 1
Loss: 0.8419 | Epoch avg loss: 0.6606931119010999
Epoch: 2
Loss: 0.7960 | Epoch avg loss: 0.6225808357390074
Epoch: 3
Loss: 0.6269 | Epoch avg loss: 0.60518131720332
Epoch: 4
Loss: 0.9415 | Epoch avg loss: 0.5979463781874913
Epoch: 5
Loss: 0.7293 | Epoch avg loss: 0.5583239945654686
Epoch: 6
Loss: 0.8130 | Epoch avg loss: 0.5512092996102113
Epoch: 7
Loss: 1.0490 | Epoch avg loss: 0.5504960492253304
Epoch: 8
Loss: 0.5549 | Epoch avg loss: 0.4990396923743762
Epoch: 9
Loss: 1.6950 | Epoch avg loss: 0.49213078532081383
Epoch: 10
Loss: 0.5844 | Epoch avg loss: 0.4568781689382516
Epoch: 11
Loss: 1.2811 | Epoch avg loss: 0.4405948751820968
Epoch: 12
Loss: 0.8421 | Epoch avg loss: 0.44852403379403627
Epoch: 13
Loss: 0.6997 | Epoch avg loss: 0.4252408771560742
Epoch: 14
Loss: 0.5928 | Epoch avg loss: 0.3874938808954679
Epoch: 15
Loss: 0.5997 | Epoch avg loss: 0.37632392447155255
Epoch: 16
Loss: 0.4040 | Epoch avg loss: 0.359555999246927

#### <a id='toc6_1_3_4_'></a>[Deeplab + Resnet101](#toc0_)

In [ ]:
selected_model_index = 3
model = model_list[selected_model_index]
segmentation_model_name = model_name_list[selected_model_index]
encoder_name = 'resnet50'if selected_model_index<2 else 'resnet101'
print(segmentation_model_name, encoder_name)

# Loss: CE + Dice (multiclass)
dice = smp.losses.DiceLoss(mode="multiclass")
ce   = nn.CrossEntropyLoss()   # add class weights if needed
def loss_fn(logits, y): return 0.5*ce(logits, y) + 0.5*dice(logits, y)

# Optim & sched
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)

# Mixed precision training loop (sketch)
scaler = torch.amp.GradScaler(DEVICE.type)

for epoch in range(EPOCHS):
    print(f'Epoch: {epoch}')
    epoch_loss = []
    model.train()

    for x, y in loader:       # x:(B,4,64,64) in [-1,1], y:(B,64,64) long
        x, y = x.clone().detach().to(DEVICE), y.clone().detach().long().to(DEVICE)

        with torch.cuda.amp.autocast(DEVICE):
            logits = model(x)
            loss = loss_fn(logits, y)
            epoch_loss.append(loss.item())
        
        
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad()

    print(f'Loss: {loss.item():.4f} | Epoch avg loss: {sum(epoch_loss)/len(epoch_loss)}')
    sched.step()

save_models(model,
            opt, 
            sched, 
            scaler, 
            (epoch+1), 
            segmentation_model_name, 
            encoder_name, 
            classification_model_path)


deeplab_r101 resnet101
Epoch: 0


C:\Users\2405647\AppData\Local\Temp\ipykernel_25828\3619364156.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(DEVICE):


Loss: 1.0897 | Epoch avg loss: 1.2606734031668076
Epoch: 1
Loss: 1.3964 | Epoch avg loss: 0.9629291932170208
Epoch: 2
Loss: 1.0794 | Epoch avg loss: 0.8979588666787515
Epoch: 3
Loss: 1.2103 | Epoch avg loss: 0.8230557051988748
Epoch: 4
Loss: 1.1453 | Epoch avg loss: 0.7854255093978002
Epoch: 5
Loss: 1.1500 | Epoch avg loss: 0.7478203303538836
Epoch: 6
Loss: 0.6540 | Epoch avg loss: 0.7177930835347909
Epoch: 7
Loss: 1.2313 | Epoch avg loss: 0.7115607917881929
Epoch: 8
Loss: 0.8636 | Epoch avg loss: 0.7146779745817184
Epoch: 9
Loss: 0.9498 | Epoch avg loss: 0.6637271785965333
Epoch: 10
Loss: 0.7598 | Epoch avg loss: 0.6626916748399918
Epoch: 11
Loss: 0.7194 | Epoch avg loss: 0.6331441749173862
Epoch: 12
Loss: 0.5826 | Epoch avg loss: 0.6173444052155201
Epoch: 13
Loss: 1.3080 | Epoch avg loss: 0.6029339011472005
Epoch: 14
Loss: 1.0344 | Epoch avg loss: 0.5833429983602121
Epoch: 15
Loss: 0.6511 | Epoch avg loss: 0.5517051111047084
Epoch: 16
Loss: 0.6683 | Epoch avg loss: 0.5436346060954608

# <a id='toc7_'></a>[Traditional](#toc0_)

### <a id='toc7_1_1_'></a>[Update path](#toc0_)

In [ ]:
classification_model_path = "../Classification models/Spatial/Traditional only"


### <a id='toc7_1_2_'></a>[Load data](#toc0_)

In [ ]:
data = np.load('../Dataset/s2-utm-33N-18E-242N-2018/DS3/7_original_data_64X64.npy')
labels = np.load('../Dataset/s2-utm-33N-18E-242N-2018/DS3/8_original_labels_64X64.npy')

labels_upadated = np.where(labels == -1, 0, labels)
train_data, validation_data, train_labels, validation_labels = train_test_split(data, labels_upadated, random_state= 2) 

image_data = np.transpose(train_data, (0, 3, 1, 2))  # (N, 4, 64, 64)

dataset = ImageLabelDataset(image_data, train_labels)
loader = DataLoader(dataset, batch_size=32, shuffle=True)


### <a id='toc7_1_3_'></a>[Train models](#toc0_)

#### <a id='toc7_1_3_1_'></a>[Unet + Resnet50](#toc0_)

In [ ]:
selected_model_index = 0
model = model_list[selected_model_index]
segmentation_model_name = model_name_list[selected_model_index]
encoder_name = 'resnet50'if selected_model_index<2 else 'resnet101'
print(segmentation_model_name, encoder_name)

# Loss: CE + Dice (multiclass)
dice = smp.losses.DiceLoss(mode="multiclass")
ce   = nn.CrossEntropyLoss()   # add class weights if needed
def loss_fn(logits, y): return 0.5*ce(logits, y) + 0.5*dice(logits, y)

# Optim & sched
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)

# Mixed precision training loop (sketch)
scaler = torch.amp.GradScaler(DEVICE.type)

for epoch in range(EPOCHS):
    print(f'Epoch: {epoch}')
    epoch_loss = []
    model.train()

    for x, y in loader:       # x:(B,4,64,64) in [-1,1], y:(B,64,64) long
        x, y = x.clone().detach().to(DEVICE), y.clone().detach().long().to(DEVICE)

        with torch.cuda.amp.autocast(DEVICE):
            logits = model(x)
            loss = loss_fn(logits, y)
            epoch_loss.append(loss.item())
        
        
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad()

    print(f'Loss: {loss.item():.4f} | Epoch avg loss: {sum(epoch_loss)/len(epoch_loss)}')
    sched.step()

save_models(model,
            opt, 
            sched, 
            scaler, 
            (epoch+1), 
            segmentation_model_name, 
            encoder_name, 
            classification_model_path)


unet_r50 resnet50
Epoch: 0


C:\Users\2405647\AppData\Local\Temp\ipykernel_25828\2639090813.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(DEVICE):


#### <a id='toc7_1_3_2_'></a>[Deepab + Resnet50](#toc0_)

In [ ]:
selected_model_index = 1
model = model_list[selected_model_index]
segmentation_model_name = model_name_list[selected_model_index]
encoder_name = 'resnet50'if selected_model_index<2 else 'resnet101'
print(segmentation_model_name, encoder_name)

# Loss: CE + Dice (multiclass)
dice = smp.losses.DiceLoss(mode="multiclass")
ce   = nn.CrossEntropyLoss()   # add class weights if needed
def loss_fn(logits, y): return 0.5*ce(logits, y) + 0.5*dice(logits, y)

# Optim & sched
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)

# Mixed precision training loop (sketch)
scaler = torch.amp.GradScaler(DEVICE.type)

for epoch in range(EPOCHS):
    print(f'Epoch: {epoch}')
    epoch_loss = []
    model.train()

    for x, y in loader:       # x:(B,4,64,64) in [-1,1], y:(B,64,64) long
        x, y = x.clone().detach().to(DEVICE), y.clone().detach().long().to(DEVICE)

        with torch.cuda.amp.autocast(DEVICE):
            logits = model(x)
            loss = loss_fn(logits, y)
            epoch_loss.append(loss.item())
        
        
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad()

    print(f'Loss: {loss.item():.4f} | Epoch avg loss: {sum(epoch_loss)/len(epoch_loss)}')
    sched.step()

save_models(model,
            opt, 
            sched, 
            scaler, 
            (epoch+1), 
            segmentation_model_name, 
            encoder_name, 
            classification_model_path)


#### <a id='toc7_1_3_3_'></a>[Unet + Resnet101](#toc0_)

In [ ]:
selected_model_index = 2
model = model_list[selected_model_index]
segmentation_model_name = model_name_list[selected_model_index]
encoder_name = 'resnet50'if selected_model_index<2 else 'resnet101'
print(segmentation_model_name, encoder_name)

# Loss: CE + Dice (multiclass)
dice = smp.losses.DiceLoss(mode="multiclass")
ce   = nn.CrossEntropyLoss()   # add class weights if needed
def loss_fn(logits, y): return 0.5*ce(logits, y) + 0.5*dice(logits, y)

# Optim & sched
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)

# Mixed precision training loop (sketch)
scaler = torch.amp.GradScaler(DEVICE.type)

for epoch in range(EPOCHS):
    print(f'Epoch: {epoch}')
    epoch_loss = []
    model.train()

    for x, y in loader:       # x:(B,4,64,64) in [-1,1], y:(B,64,64) long
        x, y = x.clone().detach().to(DEVICE), y.clone().detach().long().to(DEVICE)

        with torch.cuda.amp.autocast(DEVICE):
            logits = model(x)
            loss = loss_fn(logits, y)
            epoch_loss.append(loss.item())
        
        
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad()

    print(f'Loss: {loss.item():.4f} | Epoch avg loss: {sum(epoch_loss)/len(epoch_loss)}')
    sched.step()

save_models(model,
            opt, 
            sched, 
            scaler, 
            (epoch+1), 
            segmentation_model_name, 
            encoder_name, 
            classification_model_path)


#### <a id='toc7_1_3_4_'></a>[Deeplab + Resnet101](#toc0_)

In [ ]:
selected_model_index = 3
model = model_list[selected_model_index]
segmentation_model_name = model_name_list[selected_model_index]
encoder_name = 'resnet50'if selected_model_index<2 else 'resnet101'
print(segmentation_model_name, encoder_name)

# Loss: CE + Dice (multiclass)
dice = smp.losses.DiceLoss(mode="multiclass")
ce   = nn.CrossEntropyLoss()   # add class weights if needed
def loss_fn(logits, y): return 0.5*ce(logits, y) + 0.5*dice(logits, y)

# Optim & sched
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)

# Mixed precision training loop (sketch)
scaler = torch.amp.GradScaler(DEVICE.type)

for epoch in range(EPOCHS):
    print(f'Epoch: {epoch}')
    epoch_loss = []
    model.train()

    for x, y in loader:       # x:(B,4,64,64) in [-1,1], y:(B,64,64) long
        x, y = x.clone().detach().to(DEVICE), y.clone().detach().long().to(DEVICE)

        with torch.cuda.amp.autocast(DEVICE):
            logits = model(x)
            loss = loss_fn(logits, y)
            epoch_loss.append(loss.item())
        
        
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad()

    print(f'Loss: {loss.item():.4f} | Epoch avg loss: {sum(epoch_loss)/len(epoch_loss)}')
    sched.step()

save_models(model,
            opt, 
            sched, 
            scaler, 
            (epoch+1), 
            segmentation_model_name, 
            encoder_name, 
            classification_model_path)


# <a id='toc8_'></a>[Traditional + Sythetic](#toc0_)

### <a id='toc8_1_1_'></a>[Update path](#toc0_)

In [ ]:
classification_model_path = "../Classification models/Spatial/Traditional and GAN based"


### <a id='toc8_1_2_'></a>[Load data](#toc0_)

In [ ]:
data = np.load('../Dataset/s2-utm-33N-18E-242N-2018/DS3/11_final_gan_data_without_mixed_patches_64X64_majority50.npy')
labels = np.load('../Dataset/s2-utm-33N-18E-242N-2018/DS3/12_final_gan_labels_without_mixed_patches_64X64_majority50.npy')

labels_upadated = np.where(labels == -1, 0, labels)
train_data, validation_data, train_labels, validation_labels = train_test_split(data, labels_upadated, random_state= 2) 

image_data = np.transpose(train_data, (0, 3, 1, 2))  # (N, 4, 64, 64)

dataset = ImageLabelDataset(image_data, train_labels)
loader = DataLoader(dataset, batch_size=32, shuffle=True)


### <a id='toc8_1_3_'></a>[Train models](#toc0_)

#### <a id='toc8_1_3_1_'></a>[Unet + Resnet50](#toc0_)

In [ ]:
selected_model_index = 0
model = model_list[selected_model_index]
segmentation_model_name = model_name_list[selected_model_index]
encoder_name = 'resnet50'if selected_model_index<2 else 'resnet101'
print(segmentation_model_name, encoder_name)

# Loss: CE + Dice (multiclass)
dice = smp.losses.DiceLoss(mode="multiclass")
ce   = nn.CrossEntropyLoss()   # add class weights if needed
def loss_fn(logits, y): return 0.5*ce(logits, y) + 0.5*dice(logits, y)

# Optim & sched
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)

# Mixed precision training loop (sketch)
scaler = torch.amp.GradScaler(DEVICE.type)

for epoch in range(EPOCHS):
    print(f'Epoch: {epoch}')
    epoch_loss = []
    model.train()

    for x, y in loader:       # x:(B,4,64,64) in [-1,1], y:(B,64,64) long
        x, y = x.clone().detach().to(DEVICE), y.clone().detach().long().to(DEVICE)

        with torch.cuda.amp.autocast(DEVICE):
            logits = model(x)
            loss = loss_fn(logits, y)
            epoch_loss.append(loss.item())
        
        
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad()

    print(f'Loss: {loss.item():.4f} | Epoch avg loss: {sum(epoch_loss)/len(epoch_loss)}')
    sched.step()

save_models(model,
            opt, 
            sched, 
            scaler, 
            (epoch+1), 
            segmentation_model_name, 
            encoder_name, 
            classification_model_path)


unet_r50 resnet50
Epoch: 0


C:\Users\2405647\AppData\Local\Temp\ipykernel_25828\2639090813.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(DEVICE):


Loss: 0.5315 | Epoch avg loss: 0.34593169992932904
Epoch: 1
Loss: 0.3558 | Epoch avg loss: 0.33883213409437585
Epoch: 2
Loss: 1.1012 | Epoch avg loss: 0.35438646376132965
Epoch: 3
Loss: 0.9696 | Epoch avg loss: 0.34501570276916027
Epoch: 4
Loss: 0.5591 | Epoch avg loss: 0.33632322916617763
Epoch: 5
Loss: 1.1281 | Epoch avg loss: 0.3371298833248707
Epoch: 6
Loss: 0.8369 | Epoch avg loss: 0.3236464620209657
Epoch: 7
Loss: 1.3092 | Epoch avg loss: 0.2876975837235267
Epoch: 8
Loss: 0.8195 | Epoch avg loss: 0.2862538511936481
Epoch: 9
Loss: 0.3687 | Epoch avg loss: 0.27937003038823605
Epoch: 10
Loss: 0.5336 | Epoch avg loss: 0.2720044322598439
Epoch: 11
Loss: 0.7937 | Epoch avg loss: 0.27317150538930524
Epoch: 12
Loss: 0.7281 | Epoch avg loss: 0.2643966819517888
Epoch: 13
Loss: 0.7295 | Epoch avg loss: 0.23576915908891422
Epoch: 14
Loss: 0.5131 | Epoch avg loss: 0.23775177663908556
Epoch: 15
Loss: 0.8669 | Epoch avg loss: 0.21884507227402467
Epoch: 16
Loss: 1.9872 | Epoch avg loss: 0.225151

#### <a id='toc8_1_3_2_'></a>[Deepab + Resnet50](#toc0_)

In [ ]:
selected_model_index = 1
model = model_list[selected_model_index]
segmentation_model_name = model_name_list[selected_model_index]
encoder_name = 'resnet50'if selected_model_index<2 else 'resnet101'
print(segmentation_model_name, encoder_name)

# Loss: CE + Dice (multiclass)
dice = smp.losses.DiceLoss(mode="multiclass")
ce   = nn.CrossEntropyLoss()   # add class weights if needed
def loss_fn(logits, y): return 0.5*ce(logits, y) + 0.5*dice(logits, y)

# Optim & sched
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)

# Mixed precision training loop (sketch)
scaler = torch.amp.GradScaler(DEVICE.type)

for epoch in range(EPOCHS):
    print(f'Epoch: {epoch}')
    epoch_loss = []
    model.train()

    for x, y in loader:       # x:(B,4,64,64) in [-1,1], y:(B,64,64) long
        x, y = x.clone().detach().to(DEVICE), y.clone().detach().long().to(DEVICE)

        with torch.cuda.amp.autocast(DEVICE):
            logits = model(x)
            loss = loss_fn(logits, y)
            epoch_loss.append(loss.item())
        
        
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad()

    print(f'Loss: {loss.item():.4f} | Epoch avg loss: {sum(epoch_loss)/len(epoch_loss)}')
    sched.step()

save_models(model,
            opt, 
            sched, 
            scaler, 
            (epoch+1), 
            segmentation_model_name, 
            encoder_name, 
            classification_model_path)


deeplab_r50 resnet50
Epoch: 0


C:\Users\2405647\AppData\Local\Temp\ipykernel_25828\1448462798.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(DEVICE):


Loss: 0.9698 | Epoch avg loss: 1.2127474191097112
Epoch: 1
Loss: 0.8962 | Epoch avg loss: 0.9313050869565743
Epoch: 2
Loss: 0.8609 | Epoch avg loss: 0.8239681210655433
Epoch: 3
Loss: 1.4193 | Epoch avg loss: 0.7814949601888657
Epoch: 4
Loss: 1.5538 | Epoch avg loss: 0.7525860008138877
Epoch: 5
Loss: 1.0879 | Epoch avg loss: 0.7072269028195968
Epoch: 6
Loss: 1.0896 | Epoch avg loss: 0.6913020029090918
Epoch: 7
Loss: 0.9134 | Epoch avg loss: 0.6601533511510262
Epoch: 8
Loss: 0.6229 | Epoch avg loss: 0.6648639589548111
Epoch: 9
Loss: 0.6052 | Epoch avg loss: 0.6089386604726315
Epoch: 10
Loss: 1.5692 | Epoch avg loss: 0.5771394406373684
Epoch: 11
Loss: 0.7727 | Epoch avg loss: 0.5857622193602415
Epoch: 12
Loss: 0.7559 | Epoch avg loss: 0.5503796144173696
Epoch: 13
Loss: 1.0256 | Epoch avg loss: 0.5290318693105991
Epoch: 14
Loss: 0.6261 | Epoch avg loss: 0.4915745034813881
Epoch: 15
Loss: 0.4574 | Epoch avg loss: 0.4774923717173246
Epoch: 16
Loss: 0.4507 | Epoch avg loss: 0.4515345093722527

#### <a id='toc8_1_3_3_'></a>[Unet + Resnet101](#toc0_)

In [ ]:
selected_model_index = 2
model = model_list[selected_model_index]
segmentation_model_name = model_name_list[selected_model_index]
encoder_name = 'resnet50'if selected_model_index<2 else 'resnet101'
print(segmentation_model_name, encoder_name)

# Loss: CE + Dice (multiclass)
dice = smp.losses.DiceLoss(mode="multiclass")
ce   = nn.CrossEntropyLoss()   # add class weights if needed
def loss_fn(logits, y): return 0.5*ce(logits, y) + 0.5*dice(logits, y)

# Optim & sched
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)

# Mixed precision training loop (sketch)
scaler = torch.amp.GradScaler(DEVICE.type)

for epoch in range(EPOCHS):
    print(f'Epoch: {epoch}')
    epoch_loss = []
    model.train()

    for x, y in loader:       # x:(B,4,64,64) in [-1,1], y:(B,64,64) long
        x, y = x.clone().detach().to(DEVICE), y.clone().detach().long().to(DEVICE)

        with torch.cuda.amp.autocast(DEVICE):
            logits = model(x)
            loss = loss_fn(logits, y)
            epoch_loss.append(loss.item())
        
        
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad()

    print(f'Loss: {loss.item():.4f} | Epoch avg loss: {sum(epoch_loss)/len(epoch_loss)}')
    sched.step()

save_models(model,
            opt, 
            sched, 
            scaler, 
            (epoch+1), 
            segmentation_model_name, 
            encoder_name, 
            classification_model_path)


unet_r101 resnet101
Epoch: 0


C:\Users\2405647\AppData\Local\Temp\ipykernel_25828\3473791458.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(DEVICE):


Loss: 1.0182 | Epoch avg loss: 1.2343237778315177
Epoch: 1
Loss: 0.9807 | Epoch avg loss: 0.9425937092074981
Epoch: 2
Loss: 1.4020 | Epoch avg loss: 0.8320828584524301
Epoch: 3
Loss: 1.3236 | Epoch avg loss: 0.7531894949766306
Epoch: 4
Loss: 0.7876 | Epoch avg loss: 0.7281224389488881
Epoch: 5
Loss: 0.8929 | Epoch avg loss: 0.6673592747404025
Epoch: 6
Loss: 1.3403 | Epoch avg loss: 0.6640933107298154
Epoch: 7
Loss: 0.8148 | Epoch avg loss: 0.6232290792350585
Epoch: 8
Loss: 0.9447 | Epoch avg loss: 0.6063797216002758
Epoch: 9
Loss: 1.3825 | Epoch avg loss: 0.5792850975233775
Epoch: 10
Loss: 0.7169 | Epoch avg loss: 0.5586186051368713
Epoch: 11
Loss: 0.7518 | Epoch avg loss: 0.5183486806658598
Epoch: 12
Loss: 0.7429 | Epoch avg loss: 0.5008836566255643
Epoch: 13
Loss: 0.5327 | Epoch avg loss: 0.4939693281283745
Epoch: 14
Loss: 1.5467 | Epoch avg loss: 0.48507819725916934
Epoch: 15
Loss: 0.8243 | Epoch avg loss: 0.4350262347322244
Epoch: 16
Loss: 1.1099 | Epoch avg loss: 0.416470276335111

#### <a id='toc8_1_3_4_'></a>[Deeplab + Resnet101](#toc0_)

In [ ]:
selected_model_index = 3
model = model_list[selected_model_index]
segmentation_model_name = model_name_list[selected_model_index]
encoder_name = 'resnet50'if selected_model_index<2 else 'resnet101'
print(segmentation_model_name, encoder_name)

# Loss: CE + Dice (multiclass)
dice = smp.losses.DiceLoss(mode="multiclass")
ce   = nn.CrossEntropyLoss()   # add class weights if needed
def loss_fn(logits, y): return 0.5*ce(logits, y) + 0.5*dice(logits, y)

# Optim & sched
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)

# Mixed precision training loop (sketch)
scaler = torch.amp.GradScaler(DEVICE.type)

for epoch in range(EPOCHS):
    print(f'Epoch: {epoch}')
    epoch_loss = []
    model.train()

    for x, y in loader:       # x:(B,4,64,64) in [-1,1], y:(B,64,64) long
        x, y = x.clone().detach().to(DEVICE), y.clone().detach().long().to(DEVICE)

        with torch.cuda.amp.autocast(DEVICE):
            logits = model(x)
            loss = loss_fn(logits, y)
            epoch_loss.append(loss.item())
        
        
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad()

    print(f'Loss: {loss.item():.4f} | Epoch avg loss: {sum(epoch_loss)/len(epoch_loss)}')
    sched.step()

save_models(model,
            opt, 
            sched, 
            scaler, 
            (epoch+1), 
            segmentation_model_name, 
            encoder_name, 
            classification_model_path)


deeplab_r101 resnet101
Epoch: 0


C:\Users\2405647\AppData\Local\Temp\ipykernel_25828\3619364156.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(DEVICE):


Loss: 0.9870 | Epoch avg loss: 0.5191619605399095
Epoch: 1
Loss: 0.7310 | Epoch avg loss: 0.5196382348927168
Epoch: 2
Loss: 0.7415 | Epoch avg loss: 0.48318429176624006
Epoch: 3
Loss: 0.7048 | Epoch avg loss: 0.46006936923815656
Epoch: 4
Loss: 0.8856 | Epoch avg loss: 0.45272025007468003
Epoch: 5
Loss: 0.8068 | Epoch avg loss: 0.4210144378817998
Epoch: 6
Loss: 0.7693 | Epoch avg loss: 0.416720333437507
Epoch: 7
Loss: 0.4786 | Epoch avg loss: 0.44739814274586165
Epoch: 8
Loss: 0.3642 | Epoch avg loss: 0.40655805084567803
Epoch: 9
Loss: 0.2734 | Epoch avg loss: 0.39895409976060575
Epoch: 10
Loss: 1.7252 | Epoch avg loss: 0.4014710531784938
Epoch: 11
Loss: 0.7122 | Epoch avg loss: 0.4053614099438374
Epoch: 12
Loss: 0.4508 | Epoch avg loss: 0.37674382243018883
Epoch: 13
Loss: 0.7268 | Epoch avg loss: 0.35448307916522026
Epoch: 14
Loss: 0.4892 | Epoch avg loss: 0.3411871414058484
Epoch: 15
Loss: 0.5710 | Epoch avg loss: 0.3431276848109869
Epoch: 16
Loss: 0.3633 | Epoch avg loss: 0.369101471